In [1]:
# Save file list - adjust based on location of files

In [9]:
import os
import json
from datetime import datetime

In [28]:
# === Filename builders ===
def build_arise_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/collections/ARISE-SAI-1.5/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_ssp245_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_hist_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/development/wawg/WACCM6-TSMLT-HIST/"
        f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_g6_path(ens_num, date_range):
    base = (
        "/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/"
        f"b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base

In [35]:
# === File list generator ===
def get_file_list(scenario, ens_num):
    files = []
    num = f"{ens_num:02d}"

    if scenario == "ARISE":
        if ens_num in [5, 8, 9]:  # These files are saved yearly
            for year in range(2035, 2070 if ens_num in [8, 9] else 2069):
                start = f"{year}010100"
                end = f"{year+1}010100"
                date_range = f"{start}-{end}"
                files.append(build_arise_path(num, date_range))
            if ens_num in [5]:
                files.append(build_arise_path(num, "2069010100-2069123100"))
        else:  # These files are saved in decades
            for start, end in [(2035, 2045), (2045, 2055), (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_arise_path(num, date_range))

    elif scenario == "SSP245":
        if ens_num <= 5:  # saved through 2100
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045), (2045, 2055), (2055, 2065), (2065, 2075)]:
                date_range = f"{start}010100-{end}010100"
                files.append(build_ssp245_path(num, date_range))
        else:  # ends in 20691231
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045), (2045, 2055), (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_ssp245_path(num, date_range))

    elif scenario == "SSP245_G6":
        for start, end in [(2015, 2025), (2025, 2035), (2035, 2045), (2045, 2055), (2055, 2065), (2065, 2075), (2075, 2085)]:
            date_range = f"{start}010100-{end}010100"
            files.append(build_ssp245_path(num, date_range))

    elif scenario == "hist":
        files.append(build_hist_path(num, "1988010100-1998010100"))
        files.append(build_hist_path(num, "1998010100-1999123100"))
        files.append(build_hist_path(num, "2000010100-2010010100"))

    elif scenario == "G6-1.5K":
        for start, end in [(2035, 2045), (2045, 2055), (2055, 2065), (2065, 2075), (2075, 2084)]:
            date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2075
                    else f"{start}010100-{end}010100"
                )
            files.append(build_g6_path(num, date_range))

    return files

In [36]:
def save_file_list_with_metadata(scenario, ens_num, file_list, DIR, filename):
    data = {
        "scenario": scenario,
        "ensemble_number": ens_num,
        "generated_on": datetime.now().isoformat(),
        "files": file_list
    }
    file_path = os.path.join(DIR, filename)
    with open(file_path, "w") as f:
        json.dump(data, f, indent=2)

In [38]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/file_paths/"
SCENARIOS = ["ARISE", "SSP245", "G6-1.5K", "hist", "SSP245_G6"]
ENS_RANGE = [range(1, 11), range(1, 11), range(1, 4), range(1, 2), range(1, 4)]

for scenario, ens_range in zip(SCENARIOS, ENS_RANGE):
    for ens_num in ens_range:
        print(f"Building list of files for {scenario}, Ensemble {ens_num:02d}")
        file_list = get_file_list(scenario, ens_num)
        save_file_list_with_metadata(
            scenario,
            ens_num,
            file_list,
            SAVE_DIR,
            f"file_list_{scenario}_{ens_num}.json"
        )

Building list of files for ARISE, Ensemble 01
Building list of files for ARISE, Ensemble 02
Building list of files for ARISE, Ensemble 03
Building list of files for ARISE, Ensemble 04
Building list of files for ARISE, Ensemble 05
Building list of files for ARISE, Ensemble 06
Building list of files for ARISE, Ensemble 07
Building list of files for ARISE, Ensemble 08
Building list of files for ARISE, Ensemble 09
Building list of files for ARISE, Ensemble 10
Building list of files for SSP245, Ensemble 01
Building list of files for SSP245, Ensemble 02
Building list of files for SSP245, Ensemble 03
Building list of files for SSP245, Ensemble 04
Building list of files for SSP245, Ensemble 05
Building list of files for SSP245, Ensemble 06
Building list of files for SSP245, Ensemble 07
Building list of files for SSP245, Ensemble 08
Building list of files for SSP245, Ensemble 09
Building list of files for SSP245, Ensemble 10
Building list of files for G6-1.5K, Ensemble 01
Building list of files